Extract om2 data for autoencoder tutorial example. see here https://github.com/PaulSpence/OM2-emulator/issues/7

In [ ]:
import os
import re
import sys
import glob
import time
import yaml
import intake
import warnings
import numpy as np
import pandas as pd
import scipy as io
import xarray as xr
import matplotlib.dates as mdates

from mpi4py import MPI
from datetime import date, datetime, timedelta
from matplotlib import pyplot as plt
from dask.distributed import Client
from collections import defaultdict
from xarray.coding.times import CFDatetimeCoder

import gsw as gsw
#convert from psu to abs salinity

warnings.filterwarnings("ignore") # Suppress warnings for these docs
time_coder = CFDatetimeCoder(use_cftime=True)

In [ ]:
client = Client()
client

In [ ]:
#https://github.com/COSIMA/cosima-recipes/blob/main/01-Cooking-Lessons-101/01-Basics/01-Loading-Slicing-Dicing-Output.ipynb
catalog = intake.cat.access_nri

In [ ]:
catalog.search(name='1deg_jra55_iaf_omip2_cycle6')

In [ ]:
variables = catalog.search(name="1deg_jra55_iaf_omip2_cycle6").unique().variable
print(variables)

In [ ]:
experiment = "1deg_jra55_iaf_omip2_cycle6"
variables = ["net_sfc_heating","frazil_3d_int_z","temp","rho_dzt","rho"]

ds = catalog[experiment].search(frequency="1mon",variable=variables).to_dask(xarray_open_kwargs = dict(use_cftime=True))

In [ ]:
ds=ds.sel(time=slice('2000-01-01', '2018-12-31'))
ds

In [ ]:
area_t = catalog[experiment].search(variable="area_t").to_dask(progressbar=False)
area_t=area_t["area_t"]
area_t

In [ ]:
total_surface_heat_flx=ds["net_sfc_heating"]+ds["frazil_3d_int_z"]

In [ ]:
total_surface_heat_flx

In [ ]:
total_surface_heat_flx = total_surface_heat_flx.assign_attrs(
    long_name="Total surface heat flux including frazil ice from 1deg_jra55_iaf_omip2_cycle6",
    units=ds["net_sfc_heating"].attrs.get("units", "W m-2"),
    description="net_sfc_heating + vertically integrated frazil heat flux"
)

total_surface_heat_flx.name = "total_surface_heat_flx"
total_surface_heat_flx.to_netcdf("1deg_total_surface_heat_flx.nc")

In [ ]:
dzt=ds["rho_dzt"]/ds["rho"]
dzt[1,:,100,1].values

In [ ]:
cp=3992.10322329649
rho0=1035

In [ ]:
ocean_heat=(ds["temp"]*dzt).sum("st_ocean")*cp*rho0
ocean_heat=ocean_heat.load()

In [ ]:
ocean_heat = ocean_heat.assign_attrs(
    long_name="Vertically integrated ocean heat content from 1deg_jra55_iaf_omip2_cycle6",
    units="J/m2",
    description="((temp*rho_dzt/dzt).sum(st_ocean)*cp*rho0)"
)

ocean_heat.name = "ocean_heat_content_2d"
ocean_heat.to_netcdf("/home/561/pas561/jk72pas561/jnb/PyEarthTools/notebooks/tutorial/1deg_ocean_heat_content_2d.nc")

In [ ]:
xr.Dataset(
    {
        "area_t": area_t,
        "total_surface_heat_flx": total_surface_heat_flx,
        "ocean_heat_content_2d": ocean_heat,
    }
).to_netcdf("/home/561/pas561/jk72pas561/jnb/PyEarthTools/notebooks/tutorial/1deg_ocean_heat_emulator_data.nc")


In [ ]:
#https://access-nri-intake-catalog.readthedocs.io/en/latest/usage/quickstart.html
#From the OMIP-2 cycle6 run: /g/data/ik11/outputs/access-om2/1deg_jra55_iaf_omip2_cycle6
catalog = intake.cat.access_nri
esm_datastore = catalog.search(name="1deg_jra55_iaf_omip2_cycle6").to_source()

#esm_datastore

In [ ]:
#esm_datastore.keys()

In [ ]:
#esm_datastore.df.head()

In [ ]:
#esm_datastore.search(frequency='fx').interactive

In [ ]:
net_sfc_heat = esm_datastore.search(frequency="1mon", variable="net_sfc_heating").to_dask(progressbar=False)

In [ ]:
net_sfc_heat

In [ ]:
frazil = esm_datastore.search(frequency="1mon", variable="frazil_3d_int_z").to_dask(progressbar=False)
frazil

In [ ]:
area_t = esm_datastore.search(variable="area_t").to_dask(progressbar=False)
area_t

In [ ]:
temp = esm_datastore.search(variable="temp").to_dask(progressbar=False)
temp

In [ ]:
rho_dzt = esm_datastore.search(variable="rho_dzt").to_dask(progressbar=False)
rho_dzt

In [ ]:
rho = esm_datastore.search(variable="rho").to_dask(progressbar=False)
rho

In [ ]:
esm_datastore_filtered = esm_datastore.search(variable="net_sfc_heating")
esm_datastore_filtered.keys()

In [ ]:
dataset = esm_datastore_filtered.search(frequency="1mon").to_dask(progressbar=False)

dataset

In [ ]:
print('Starting ACCESS-OM2 and SOTS T/S analysis.')
sys.stdout.flush()

catalog = intake.cat.access_nri
#print(catalog.keys())

#----------------------------------------------------------------------------------------------
# Date of interest 
#----------------------------------------------------------------------------------------------

start_date = np.datetime64("2010-01-01")
end_date = np.datetime64("2020-12-31")

#----------------------------------------------------------------------------------------------
# Variable and degrees of interest in ACCESS-OM2 
#----------------------------------------------------------------------------------------------

experiments = ['1deg_jra55_iaf_omip2_cycle6'] #['01deg_jra55v140_iaf_cycle4', '01deg_jra55v140_iaf_cycle4_jra55v150_extension']]
#experiments = ['1deg_jra55_iaf_omip2_cycle6', ['01deg_jra55v140_iaf_cycle4', '01deg_jra55v140_iaf_cycle4_jra55v150_extension']]

#with bgc 025deg_jra55_iaf_omip2_cycle7_bgc 
#----------------------------------------------------------------------------------------------
# Variables of interest
#----------------------------------------------------------------------------------------------
#net_sfc_heating = sfc_hflux_coupler + sfc_hflux_pme + sfc_hflux_from_runoff + sfc_hflux_from_calving

#sfc_hflux_coupler = swflx + lw_heat + fprec_melt_heat + calving_melt_heat + sens_heat + evap_heat + mh_flux + liceht


var  =  ('pot_temp','salt')

#----------------------------------------------------------------------------------------------
# Region of interest
#----------------------------------------------------------------------------------------------

#region_lat   = np.array([-50,-45])                    # yt_ocean,N
#region_lon   = np.array([141,146])                    # xt_ocean,E
#region_lon   = region_lon - 360                         # -> [-213, -205]

#region = [region_lat, region_lon]

In [ ]:
#----------------------------------------------------------------------------------------------
# This section collects all the experiment / degrees / variable paths
#----------------------------------------------------------------------------------------------
print(f'Processing ACCESS {var} data...')
sys.stdout.flush()
    

path_dict = {}

catalogs = []

for exp in experiments:

    if isinstance(exp, list):  # both tenth degree + extention
        merged_name = exp[0]   
        path_dict[merged_name] = []
    
        #bug here for salt vs pot_rho
        for subexp in exp:
            filtered = catalog[subexp].search(
                variable=var,
                frequency='1mon',
                realm='ocean',
            )
            df = filtered.df
            df['start_date'] = pd.to_datetime(df['start_date'])
            df['end_date']   = pd.to_datetime(df['end_date'])
    
            df = df[
                (df['start_date'] >= np.datetime64("2010-01-01")) &
                (df['end_date']   <= np.datetime64("2023-12-31"))
            ]
    
            file_list = df['path'].tolist()
            path_dict[merged_name].extend(file_list)
    
        print(f"{merged_name}: {len(path_dict[merged_name])} files")
    
    else:  
        path_dict[exp] = []
        filtered = catalog[exp].search(
            variable=var,
            frequency='1mon',
            realm='ocean',
        )
        df = filtered.df
        df['start_date'] = pd.to_datetime(df['start_date'])
        df['end_date']   = pd.to_datetime(df['end_date'])
    
        df = df[
            (df['start_date'] >= np.datetime64("2010-01-01")) &
            (df['end_date']   <= np.datetime64("2023-12-31"))
        ]
    
        file_list = df['path'].tolist()
        path_dict[exp].extend(file_list)
    
        print(f"{exp}: {len(file_list)} files")
    